In [ ]:
import re
import unicodedata
import pandas as pd

# ── Normalisation ─────────────────────────────────────────────────────────────
def _normalize(s):
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[\u0027\u2018\u2019\u201a\u2032]", "'", s)
    return re.sub(r"\s+", " ", s).casefold().strip()

def is_full_caps_line(line):
    letters = [ch for ch in line if ch.isalpha()]
    return len(letters) >= 2 and all(ch.isupper() for ch in letters)

# ── Statuts ───────────────────────────────────────────────────────────────────
ALLOWED_STATUSES = {"applicable", "non applicable", "not applicable",
                    "sans objet", "nicht anwendbar", "anwendbar"}

STATUS_TO_BOOL = {
    "applicable": True, "anwendbar": True,
    "non applicable": False, "not applicable": False,
    "sans objet": False, "nicht anwendbar": False,
}

def extract_status(line):
    n = _normalize(line)
    for s in sorted(ALLOWED_STATUSES, key=len, reverse=True):
        if n.endswith(s):
            return s
    return None

# ── Zones & personnes ─────────────────────────────────────────────────────────
PERSON_TYPES = [
    ("personne",     re.compile(r"Personnes?\b|Persons?\b|Privatpersonen\b|Non.Natural|Non.Retail|Clients?\b|Nat[u\u00fc]rliche\b", re.I)),
    ("investisseur", re.compile(r"Investisseurs?|Investors?\b|Privatinvestoren\b|Retail\b", re.I)),
]

ZONES = [
    ("EEE",         re.compile(r"\bEEE\b|\bEEA\b|\bEWR\b", re.I)),
    ("Royaume-Uni", re.compile(r"Royaume[\s\-\n]+Uni|United\s+Kingdom|\bUK\b|K[o\u00f6]nigreich", re.I)),
    ("Suisse",      re.compile(r"\bSuisse\b|\bSwiss\b|\bSchweiz\b", re.I)),
]

# ── Regex de section ──────────────────────────────────────────────────────────
cleanup_re     = re.compile(r"DEFINITIVES\s+APPLICABLES|APPLICABLE\s+FINAL|ANWENDBARE\s+ENDG[U\u00dc]LTIGE", re.I)
start_re       = re.compile(r"(?m)^\s*\d+\s*\.\s*(?:PLACEMENT|DISTRIBUTION|PLATZIERUNG)\b", re.I)
end_re         = re.compile(r"(?m)^\s*\d+\s*\.\s*(?:MODALIT(?:ES|\u00c9S)\s+DE\s+L[\u0027\u2018\u2019]OFFRE|TERMS\s+AND\s+CONDITIONS\s+OF\s+THE\s+OFFER|EMISSIONSBEDINGUNGEN\s+DES\s+ANGEBOTS)", re.I)
interdiction_re= re.compile(r"(?:Interdiction\s+de\s+Ventes|Prohibition\s+of\b|Verkaufsverbot)\b", re.I)

MEANINGFUL_RE  = re.compile(
    r"Interdiction\s+de\s+Ventes|Prohibition\s+of\b|Verkaufsverbot"
    r"|Investisseurs?|Investors?\b|Privatinvestoren|Personnes?\b|Persons?\b"
    r"|Privatpersonen\b|Non[\s\-]Natural|Non[\s\-]Retail|Clients?\b|Nat[u\u00fc]rliche\b"
    r"|D[\u00e9e]tail\b|\bEEE\b|\bEEA\b|\bEWR\b|\bUK\b|United\s+Kingdom"
    r"|Royaume[\s\-\n]+Uni|K[o\u00f6]nigreich|\bSuisse\b|\bSwiss\b|\bSchweiz\b"
    r"|\bApplicable\b|\bAnwendbar\b|Non\s+Applicable|Not\s+Applicable|Nicht\s+Anwendbar|Sans\s+Objet",
    re.I,
)

def trim_to_last_meaningful(text):
    lines = text.splitlines()
    last_idx = 0
    for i, line in enumerate(lines):
        if MEANINGFUL_RE.search(line):
            last_idx = i
    return "\n".join(lines[: last_idx + 1]).strip()

# ── Parsing des blocs ─────────────────────────────────────────────────────────
def parse_blocks(text):
    lines = text.splitlines()
    status_positions = [i for i, l in enumerate(lines) if extract_status(l) is not None]
    tuples = []
    for idx, start in enumerate(status_positions):
        end        = status_positions[idx + 1] if idx + 1 < len(status_positions) else len(lines)
        block_text = "\n".join(lines[start:end])
        person = next((name for name, pat in PERSON_TYPES if pat.search(block_text)), "inconnu")
        zone   = next((name for name, pat in ZONES        if pat.search(block_text)), "inconnu")
        tuples.append((person, zone, extract_status(lines[start])))
    for i in range(1, len(tuples)):
        p, z, s = tuples[i]
        if z == "inconnu" and tuples[i - 1][1] != "inconnu":
            tuples[i] = (p, tuples[i - 1][1], s)
    lookup = {}
    for p, z, s in tuples:
        lookup.setdefault(f"{p}-{z}", []).append(s)
    return lookup

def resolve(lookup, key):
    vals = lookup.get(key, [])
    if len(vals) != 1:
        return "Error"
    return STATUS_TO_BOOL.get(vals[0], "Error")

# ── Pipeline principal (texte → résultat) ─────────────────────────────────────
def process_text(pdf_name, raw_text):
    ERROR = dict(pdf_name=pdf_name, text_concerned="", EEA="Error", UK="Error", Swiss="Error")
    text  = cleanup_re.sub("", raw_text)
    m_start = start_re.search(text)
    if not m_start:
        return {**ERROR, "text_concerned": "[section introuvable]"}
    m_end = end_re.search(text, pos=m_start.end())
    if not m_end:
        return {**ERROR, "text_concerned": "[fin de section introuvable]"}
    section = text[m_start.start():m_end.start()].strip()
    m_int   = interdiction_re.search(section)
    if not m_int:
        return {**ERROR, "text_concerned": "[interdiction introuvable]"}
    section = section[m_int.start():].strip()
    section = "\n".join(l for l in section.splitlines() if not is_full_caps_line(l)).strip()
    section = trim_to_last_meaningful(section)
    lookup  = parse_blocks(section)
    return dict(
        pdf_name      = pdf_name,
        text_concerned= section,
        EEA           = resolve(lookup, "personne-EEE"),
        UK            = resolve(lookup, "personne-Royaume-Uni"),
        Swiss         = resolve(lookup, "personne-Suisse"),
    )

print("Helpers chargés.")

In [ ]:
rows = []

for _, row in df_sg.iterrows():
    pdf_name = row["pdf_name"]
    try:
        content = row["content"] if isinstance(row["content"], str) else ""
        rows.append(process_text(pdf_name, content))
    except Exception as e:
        print(f"Erreur sur {pdf_name} : {e}")
        rows.append(dict(pdf_name=pdf_name, text_concerned="[erreur]", EEA="Error", UK="Error", Swiss="Error"))

resultat = pd.DataFrame(rows, columns=["pdf_name", "text_concerned", "EEA", "UK", "Swiss"])
resultat